In [2]:
# Import for data Manipulation and Representation
import matplotlib.pyplot         as plt
import pandas                    as pd
import numpy                     as np
import seaborn                   as sns
import plotly.express            as px
import os
from sklearn.model_selection    import train_test_split
from sklearn.decomposition      import PCA

# Import Classifiers
from sklearn.neural_network     import MLPClassifier
from sklearn.metrics            import roc_auc_score, accuracy_score, recall_score, confusion_matrix
from joblib                     import dump, load
from sklearn.svm                import SVC
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.ensemble           import AdaBoostClassifier,RandomForestClassifier
from sklearn.tree               import DecisionTreeClassifier
from sklearn.linear_model       import LogisticRegression

# Import Classifiers Metrics
from sklearn.metrics            import roc_curve,auc
from sklearn.model_selection    import KFold,GridSearchCV, RandomizedSearchCV,cross_val_score

## Useful Variables:
- <code>all_best_clfs</code> -> Dictionary containing all trained classifier (CLF_NAME : BEST_CLF)
- <code>use_pca</code> -> Flag for PCA analysis or Statistic Analisys on Feature Selection
- <code>colums</code> -> Names for the binary labels
- <code>colormap</code> -> Colormap for all the confusion matrices

In [3]:
all_best_clfs = dict()
use_pca       = True
columns       = ['No Stress','Stress']
colormap      = 'Blues'

## Utility function for Confusion Matrix plot

In [4]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / np.sum(cm).astype('float')
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

## Load Data

In [5]:
train_data = pd.read_csv(r"D:\Jacopo\PhD\Applications\rhi_data_w_eeg\fold_1_train.csv")
test_data = pd.read_csv(r"D:\Jacopo\PhD\Applications\rhi_data_w_eeg\fold_1_test.csv")

train_target = pd.read_csv(r"D:\Jacopo\PhD\Applications\rhi_data_w_eeg\train_fold_1_label.csv")
test_target  = pd.read_csv(r"D:\Jacopo\PhD\Applications\rhi_data_w_eeg\test_fold_1_label.csv")


print('Dimensioni di x_train:', train_data.shape)
print('Dimensioni di train_target:', train_target.shape)

Dimensioni di x_train: (136, 275)
Dimensioni di train_target: (136, 1)


## Dataset Standardisation

In [6]:
train_data_std = pd.DataFrame()
test_data_std  = pd.DataFrame()

for col in train_data.columns:
  if col == "Index":
    continue
  mu  = train_data[col].mean()
  std = train_data[col].std() 
  train_data_std[col] = train_data[col].apply(lambda x: (x - mu)/std)
  test_data_std[col]  = test_data[col].apply(lambda x: (x - mu)/std)

C:\Users\jacop\AppData\Local\Temp\ipykernel_3920\913469322.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_data_std[col] = train_data[col].apply(lambda x: (x - mu)/std)
C:\Users\jacop\AppData\Local\Temp\ipykernel_3920\913469322.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_data_std[col]  = test_data[col].apply(lambda x: (x - mu)/std)
C:\Users\jacop\AppData\Local\Temp\ipykernel_3920\913469322.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

## Features Correlation plot
Place <code>debug = True </code> for visualising feature correlation plot

In [7]:
debug=False
if debug:
  f, ax= plt.subplots(figsize=(30,30))
  sns.heatmap(data.corr(),annot=True)
  pd.plotting.scatter_matrix(data,figsize=(25,25))

## Features Selection:
- <code> use_pca = True </code>  -> Principal Component Analysis is done
- <code> use_pca = False </code> -> Feature Selection Correlation Based is done

In [8]:
if not use_pca:
  #calcolo correlazione tra le features e le label --> imposto 0.5 come valore di correlazione
  a = train_data_std.corrwith(train_target, axis=0, method='pearson')
  #a = np.corrcoef(train_data_std,train_target,rowvar=False)
  features_selected =[]

  for i in a.index:
    if a[i] >= 0.7 or a[i] <= -0.7:
        features_selected.append(i)
  a=[]
  print(features_selected)

  train_data_features_selection = train_data_std.loc[:,features_selected]
else:
  variance=0.98
  pca = PCA(variance,random_state=42)
  pca.fit(train_data_std)

  dump(pca,'pca.joblib')

  exp_var = pca.explained_variance_ratio_
  exp_var_cumul = np.cumsum(exp_var)
  plt.title('PCA Analysis')
  plt.xlabel('# Components')
  plt.ylabel('Variance')
  plt.plot(exp_var_cumul)
  
  print(f'N componenti principali selezionate: {exp_var}')

  pca_data_train = pca.transform(train_data_std) 

  n_components = pca.n_components_
  train_data_pca = pd.DataFrame(pca_data_train, columns=['pc{}'.format(i) for i in range(1, n_components+1)])

  #applico la stessa trasformazione al test set, dopo aver addestrato il modello sui dati del training (.fit e .transform al training) 
  pca_data_test = pca.transform(test_data_std)
  test_data_pca = pd.DataFrame(pca_data_test, columns=['pc{}'.format(i) for i in range(1, n_components+1)])

  print(test_data_pca)
  pca_data_test = pca.transform(test_data_std)
  test_data_pca = pd.DataFrame(pca_data_test, columns=['pc{}'.format(i) for i in range(1, n_components+1)])

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Principal Component 1 vs 1 Scatter Plot

In [ ]:
if debug and use_pca:
    # Scatter matrix per training-set
    fig_train = px.scatter_matrix(
        train_data_pca,
        dimensions=train_data_pca.columns,
        color= train_target,
        title='PCA Scatter Matrix (Training)'
        )

    fig_train.update_traces(diagonal_visible=False)
    fig_train.show()

    # Scatter matrix per test-set
    fig_test = px.scatter_matrix(
        test_data_pca,
        dimensions=test_data_pca.columns,
        color= test_target,
        title='PCA Scatter Matrix (Test)'
        )

    fig_test.update_traces(diagonal_visible=False)
    fig_test.show()

## Retrieving most relevant features from PCA analysis

In [ ]:
# Identifichiamo le features più rilevanti
variances = pca.explained_variance_ratio_
feature_importances = pd.DataFrame(variances, index=['pc{}'.format(i) for i in range(1, n_components+1)], columns=['importance'])
feature_importances.sort_values(by='importance', ascending=False, inplace=True)

components = pca.components_
feature_weights = pd.DataFrame(components.T, index=train_data_std.columns, columns=['pc{}'.format(i) for i in range(1, n_components+1)])
feature_weights_abs = feature_weights.abs().loc[:, feature_importances.index]
feature_importances_orig = feature_weights_abs.max(axis=1)

# Ordina le feature in base all'importanza
feature_importances_orig_sorted = feature_importances_orig.sort_values(ascending=False)

# Stampa le feature e il loro peso massimo assunto tra le componenti principali selezionate con varianza > 0.98
for feature in feature_importances_orig_sorted.index:
    max_weight = feature_weights_abs.loc[feature].max()
    print(f"{feature}: {max_weight:.3f}")

print(feature_importances_orig.index[0])
print(feature_importances_orig.index[1])
print(feature_importances_orig.index[2])

## Machine Learning Section
### Seven different Classifiers Trained by using <code> Nested K-Fold Cross Validation </code>
Selected Classifier are:
- <code>Support Vector Classifier with a Gaussian Kernel</code>
- <code>Support Vector Classifier with a Polynomial Kernel</code>
- <code>Multilayer Perceptron</code>
- <code>K-Neareast Neighbour Classifier</code>
- <code>AdaBoost Classifier</code>
- <code>Logistic Regression Classifier</code>
- <code>Random Forest Classifier</code>

## Support Vector Classifier with Gaussian Kernel

In [ ]:

# K-Fold
K_outer = 5
K_inner = 2

if not os.path.exists('./svm_rbf'):
  os.mkdir('svm_rbf')

# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []

clf_counter = 1
# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)

# Parameters to be optimized
C_range = np.logspace(-2, 5, 50)
gamma_range = np.logspace(-9, 3, 50) 

# To-be Optimised Parameters with Nested K-Fold Cross Validation
param_grid = {'C': C_range, 'gamma':gamma_range} 

# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est = SVC(random_state=42, kernel='rbf')
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'Best Params:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  # Use dump function to save trained and optimised classifier 
  dump(grid,f'svm_rbf/SVM_rbf{clf_counter}.joblib')
  
  clf_counter += 1

  # Performance over Test Set 
  accuracies.append(accuracy_score(test_target, y_pred))
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))
  
  # Plot Confusion Matric for performance evaluation
  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix SVC-RBF',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies)
ax[0].set_title("Accuracies over Test Set")
ax[0].setyticks(np.arange(1,K_outer+1,1))
ax[1].plot(ticks,sensitivity)
ax[1].set_title("Sensitivities over Test Set")
ax[1].setyticks(np.arange(1,K_outer+1,1))
ax[2].plot(ticks,auc_scores)
ax[2].set_title("AUC over Test Set")
ax[2].setyticks(np.arange(1,K_outer+1,1))

# Save best classifier number over folds
best = np.argmax(accuracies) + 1
print(f'Best Classifier is n. {best}')
all_best_clfs.update({'SVM_rbf' : best})

## Support Vector Classifier with Polynomial Kernel

In [ ]:
# SVM (poly)
# K-Fold
K_outer = 5
K_inner = 2
if not os.path.exists('./svm_poly'):
  os.mkdir('svm_poly')
# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []
ticks       = np.arange(1,K_outer+1,1)
curr_acc    = 0
clf_counter = 1
# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)

# Parameters to be optimized
#C_range = np.logspace(-2, 5, 50)
#gamma_range = np.logspace(-9, 3, 50)  
C_range = np.logspace(-2, 10, 50)
gamma_range = np.logspace(-9, 3, 50) 

param_grid = {'C': C_range, 'gamma':gamma_range} 

# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est = SVC(random_state=42, kernel='poly')
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'I parametri ottimali sono:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  dump(grid,f'svm_poly/SVM_poly{clf_counter}.joblib')
  clf_counter += 1

  curr_acc = accuracy_score(test_target, y_pred)

  #performance classificatore SVM --> sul test set 
  accuracies.append(curr_acc)
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))

  print("Acc=", accuracies)
  print("Sens=", sensitivity)
  print("Auc=", auc_scores)
  
  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix SVM-poly',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies)
ax[0].set_title("Accuracies over Test Set")
ax[1].plot(ticks,sensitivity)
ax[1].set_title("Sensitivities over Test Set")
ax[2].plot(ticks,auc_scores)
ax[2].set_title("AUC over Test Set")

best = np.argmax(accuracies) + 1

print(f'Best Classifier is n. {best}')

all_best_clfs.update({'SVM_poly' : best})

## Multilayer Perceptron

In [ ]:
# K-Fold
K_outer = 5
K_inner = 2

if not os.path.exists('./mlp'):
  os.mkdir('mlp')

# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []
ticks       = np.arange(1,K_outer+1,1)

# Parameters
#hidden_layer_sizes=(20)
#learning_rate_init=0.1
#max_iter=250

hidden_layers  = [9, 8, 7, 6, 5, 4]
#,(8,6,4))
learning_range = [0.001, 0.01, 0.1]

clf_counter = 1

# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)
 

param_grid = {'learning_rate_init':learning_range, 'solver':['adam','sgd','lbfgs']} 

# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est  = MLPClassifier( hidden_layer_sizes=[9, 8, 7, 6, 5, 4], max_iter=500, random_state=42)
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'I parametri ottimali sono:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  dump(grid,f'mlp/mlp{clf_counter}.joblib')
  clf_counter += 1

  #performance classificatore SVM --> sul test set 
  accuracies.append(accuracy_score(test_target, y_pred))
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))
  
  print("Acc=", accuracies)
  print("Sens=", sensitivity)
  print("Auc=", auc_scores)
  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix MultiLayer Perceptron',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies)
ax[0].set_title("Accuracies over Test Set")
ax[1].plot(ticks,sensitivity)
ax[1].set_title("Sensitivities over Test Set")
ax[2].plot(ticks,auc_scores)
ax[2].set_title("AUC over Test Set")
best = np.argmax(accuracies) + 1

print(f'Best Classifier is n. {best}')

all_best_clfs.update({'mlp' : best})

## K-Nearest Neighbour Classifier

In [ ]:
# K-Fold
K_outer = 5
K_inner = 2

if not os.path.exists('./knn'):
  os.mkdir('knn')

# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []
ticks       = np.arange(1,K_outer+1,1)

# Parameters
n_neighbors = list(range(1, 20))

clf_counter = 1

# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)
 
param_grid = {'n_neighbors': n_neighbors, 'algorithm':['ball_tree', 'kd_tree', 'brute']}


# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est  =  KNeighborsClassifier()
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'n of neighbors ottimali sono:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  dump(grid,f'knn/knn{clf_counter}.joblib')
  clf_counter += 1

  #performance classificatore --> sul test set 
  accuracies.append(accuracy_score(test_target, y_pred))
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))
  
  print("Acc=", accuracies)
  print("Sens=", sensitivity)
  print("Auc=", auc_scores)

  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix KNN',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies)
ax[0].set_title("Accuracies over Test Set")
ax[1].plot(ticks,sensitivity)
ax[1].set_title("Sensitivities over Test Set")
ax[2].plot(ticks,auc_scores)
ax[2].set_title("AUC over Test Set")
best = np.argmax(accuracies) + 1

print(f'Best Classifier is n. {best}')

all_best_clfs.update({'knn' : best})


# AdaBoost

In [ ]:
# K-Fold
K_outer = 5
K_inner = 2

if not os.path.exists('./adaboost'):
  os.mkdir('adaboost')

# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []
ticks       = np.arange(1,K_outer+1,1)


# definizione del classificatore base
base_estimator = DecisionTreeClassifier(max_depth=1)

clf_counter = 1


# Parameters 
#learning_rate = 0.1
#n_estimators = 50 

# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)
 

param_grid = {'n_estimators': [10, 50, 100], 'learning_rate': [0.001, 0.01, 0.1, 1.0]}

# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est  = AdaBoostClassifier(estimator=base_estimator, random_state=42)
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'I parametri ottimali sono:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  dump(grid,f'adaboost/adaboost{clf_counter}.joblib')
  clf_counter += 1


  #performance classificatore  --> sul test set 
  accuracies.append(accuracy_score(test_target, y_pred))
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))
  
  print("Acc=", accuracies)
  print("Sens=", sensitivity)
  print("Auc=", auc_scores)

  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix AdaBoost',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies,'o')
ax[0].set_title("Accuracies over Test Set")
ax[1].plot(ticks,sensitivity,'o')
ax[1].set_title("Sensitivities over Test Set")
ax[2].plot(ticks,auc_scores,'o')
ax[2].set_title("AUC over Test Set")
best = np.argmax(accuracies) + 1

print(f'Best Classifier is n. {best}')

all_best_clfs.update({'adaboost' : best})



## Logistic Regression

In [ ]:
# K-Fold
K_outer = 5
K_inner = 2
if not os.path.exists('./log_regr'):
  os.mkdir('log_regr')


# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []
ticks       = np.arange(1,K_outer+1,1)

clf_counter = 1
# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)

# Parameters to be optimized
C_range = np.arange(1e-1,20,100)

param_grid = {'C': C_range, 'solver':['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']} 

# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est = LogisticRegression(random_state=42, max_iter=200)
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'I parametri ottimali sono:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  dump(grid,f'log_regr/log_regr{clf_counter}.joblib')
  clf_counter += 1

  #performance classificatore SVM --> sul test set 
  accuracies.append(accuracy_score(test_target, y_pred))
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))
  
  print("Acc=", accuracies)
  print("Sens=", sensitivity)
  print("Auc=", auc_scores)

  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix Logistic Regression',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies,'o')
ax[0].set_title("Accuracies over Test Set")
ax[1].plot(ticks,sensitivity,'o')
ax[1].set_title("Sensitivities over Test Set")
ax[2].plot(ticks,auc_scores,'o')
ax[2].set_title("AUC over Test Set")
best = np.argmax(accuracies) + 1

print(f'Best Classifier is n. {best}')

all_best_clfs.update({'log_regr' : best})

## Random Forest Classifier

In [ ]:
# K-Fold
K_outer = 5
K_inner = 2

if not os.path.exists('./rdm_frs'):
  os.mkdir('rdm_frs')

# Metrics
accuracies  = []
sensitivity = []
auc_scores  = []
ticks       = np.arange(1,K_outer+1,1)

clf_counter = 1

# Data
x_train = pca_data_train if use_pca else train_data_features_selection
labels  = train_target.to_numpy()
x_test  = pca_data_test if use_pca else test_data_features_selection

# Nested 5-Fold Cross Validation for Parameters Optimization
inner_cv = KFold(n_splits=K_inner,shuffle=True,random_state=42)
outer_cv = KFold(n_splits=K_outer,shuffle=True,random_state=42)

# Parameters to be optimized
n_estimators_range = range(30,200,10)

param_grid = {'n_estimators': n_estimators_range, 'criterion':['gini', 'entropy', 'log_loss']} 

# Train/Test loop
for train_index, valid_index in outer_cv.split(x_train, train_target):
  
  # Outer Fold Split
  X_train, X_valid = x_train[train_index], x_train[valid_index]
  y_train, y_valid = labels[train_index],  labels[valid_index]

  # Estimator
  est = RandomForestClassifier(random_state=42)
  grid = GridSearchCV(est, param_grid, scoring = 'accuracy', cv=inner_cv) 
  
  # Train & Optimize on Inner K-Fold
  grid.fit(X_train,y_train)

  print(f'I parametri ottimali sono:{grid.best_params_}')

  y_pred = grid.predict(x_test)

  dump(grid,f'rdm_frs/rdm_frs{clf_counter}.joblib')
  clf_counter += 1


  #performance classificatore SVM --> sul test set 
  accuracies.append(accuracy_score(test_target, y_pred))
  sensitivity.append(recall_score(test_target, y_pred))
  auc_scores.append(roc_auc_score(test_target, y_pred))

  print("Acc=", accuracies)
  print("Sens=", sensitivity)
  print("Auc=", auc_scores)
  
  plot_confusion_matrix(confusion_matrix(test_target,y_pred),columns,f'Confusion Matrix Random Forest',cmap=colormap,normalize=False)

# Performance Plots
fig,ax = plt.subplots(3,1,sharex='all',figsize=[12,8])
ax[0].plot(ticks,accuracies,'o')
ax[0].set_title("Accuracies over Test Set")
ax[1].plot(ticks,sensitivity,'o')
ax[1].set_title("Sensitivities over Test Set")
ax[2].plot(ticks,auc_scores,'o')
ax[2].set_title("AUC over Test Set")
best = np.argmax(accuracies) + 1

print(f'Best Classifier is n. {best}')

all_best_clfs.update({'rdm_frs' : best})

---
# _Test Best Classifiers over outer data from another dataset_
---

### Load new data and standardise extracted features, no need to split because this is our new test set

In [ ]:
df = pd.read_csv('sala.csv', delimiter=';')
label = df.Label.copy()

for col in df.columns:
  if col == "Index":
    continue
  mu  = df[col].mean()
  std = df[col].std() 
  df[col] = df[col].apply(lambda x: (x - mu)/std)


### Load PCA and apply over new data

In [ ]:
df = df.drop('Label', axis='columns')
pca = load('pca.joblib')
df = pca.transform(df)

### Loop over <code>.joblib</code> files loading previously trained classifiers then predict by using our new data, then write classifiers' best parameters in a new file

In [ ]:
clf_names = ['SVM_rbf','SVM_poly','mlp','knn','adaboost','log_regr','rdm_frs']

with open('best_params.txt','w+') as f:
    for clf_name in clf_names:
        
        clf = load(f'{clf_name}/{clf_name}{all_best_clfs.get(clf_name)}.joblib')
        test_pred = clf.predict(df)
        plot_confusion_matrix(confusion_matrix(label,test_pred),columns,f'Confusion Matrix {clf_name}',cmap=colormap,normalize=False)

        print(f'{clf_name} :',file=f)
        
        for key in clf.best_params_:    
            print(f'\t{key} = {clf.best_params_[key]}',file=f)
        
        print(f'\n',file=f)    

        f = plt.gcf()
        f.savefig(f'images/{clf_name}.png',dpi=300)